# Workflow gros fichiers : merge -> patcher -> split en 3 pour LiMatch

Ce notebook fait exactement l'ordre que tu veux :

1. **merge streaming** des 2 fichiers *aller* en un seul `.las`
2. **patcher** entre `aller_merged.las` et le fichier *retour*
3. **split spatial en 3** des 2 fichiers de superposition produits par patcher
4. sortie prête pour lancer LiMatch part par part

Le code est pensé pour **éviter d'exploser la RAM** :
- lecture LAS **par chunks**
- pas de chargement complet en mémoire
- pas de `header.copy()` (on utilise `copy.deepcopy`)

In [1]:
from pathlib import Path
import copy
import os
import time
import numpy as np
import pandas as pd
import laspy

# ============================================================
# USER CONFIG
# ============================================================

# --- fichiers input ---
ALLER_FILES = [
    Path("/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/merged/ALL/merged_1000_VUX_PUCK.las"),
    Path("/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/merged/ALL/merged_2000_VUX_PUCK.las"),
]

RETOUR_FILE = Path("/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/merged/ALL/merged_3000_VUX_PUCK.las")

# --- dossier de travail ---
WORK_ROOT = Path("/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher")
MERGE_DIR = WORK_ROOT / "merged"
CROP_DIR = WORK_ROOT / "cropped"
PATCHER_DIR = WORK_ROOT / "patcher_outputs"
SPLIT_DIR = WORK_ROOT / "split_for_limatch"

MERGE_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)
PATCHER_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

MERGED_ALLER = MERGE_DIR / "aller_merged.las"
CROPPED_ALLER = CROP_DIR / "aller_crop.las"

# --- réglages mémoire / streaming ---
CHUNK_SIZE = 2_000_000
SAMPLE_TARGET = 600_000
SAMPLE_PER_CHUNK = 20_000
N_PARTS = 3
FORCE_REBUILD = False

# ============================================================
# OUTAGE WINDOW
# ============================================================
OUTAGE_START = 305120.0
OUTAGE_END   = 305700.0
MARGIN_S = 10.0

ALLER_TMIN = OUTAGE_START - MARGIN_S   # 305110
ALLER_TMAX = OUTAGE_END   + MARGIN_S   # 305710

# --- après patcher : à renseigner plus bas ---
PATCH_A = None
PATCH_B = None

print("WORK_ROOT      :", WORK_ROOT)
print("MERGED_ALLER   :", MERGED_ALLER)
print("CROPPED_ALLER  :", CROPPED_ALLER)
print("RETOUR_FILE    :", RETOUR_FILE)
print("ALLER WINDOW   :", (ALLER_TMIN, ALLER_TMAX))

WORK_ROOT      : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher
MERGED_ALLER   : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/merged/aller_merged.las
CROPPED_ALLER  : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/cropped/aller_crop.las
RETOUR_FILE    : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/merged/ALL/merged_3000_VUX_PUCK.las
ALLER WINDOW   : (305110.0, 305710.0)


In [2]:
def sizeof_gb(path: Path) -> float:
    return path.stat().st_size / (1024**3)

def dim_names(header):
    return list(header.point_format.dimension_names)

def print_las_info(path: Path):
    with laspy.open(path) as r:
        hdr = r.header
        print(f"\n{path}")
        print(f"  point_count : {hdr.point_count:,}")
        print(f"  point_format: {hdr.point_format.id}")
        print(f"  scales      : {hdr.scales}")
        print(f"  offsets     : {hdr.offsets}")
        print(f"  file size   : {sizeof_gb(path):.2f} GB")
        print(f"  dims        : {dim_names(hdr)}")

def stream_las_time_bounds(path: Path, chunk_size=2_000_000):
    tmin = np.inf
    tmax = -np.inf
    total = 0

    with laspy.open(path) as r:
        if "gps_time" not in dim_names(r.header):
            raise ValueError(f"'gps_time' absent de {path}")

        for pts in r.chunk_iterator(chunk_size):
            gt = pts.gps_time
            if len(gt) == 0:
                continue
            tmin = min(tmin, float(np.min(gt)))
            tmax = max(tmax, float(np.max(gt)))
            total += len(gt)

    return tmin, tmax, total

def check_inputs_compatible(paths):
    with laspy.open(paths[0]) as r0:
        pf0 = r0.header.point_format.id
        dims0 = dim_names(r0.header)

    for p in paths[1:]:
        with laspy.open(p) as r:
            pf = r.header.point_format.id
            dims = dim_names(r.header)
            if pf != pf0:
                raise ValueError(f"Point format différent:\n- {paths[0]} -> {pf0}\n- {p} -> {pf}")
            if dims != dims0:
                raise ValueError(f"Dimensions différentes:\n- {paths[0]} -> {dims0}\n- {p} -> {dims}")

def remap_points_to_target_header(points, target_header):
    out = laspy.ScaleAwarePointRecord.zeros(len(points), header=target_header)

    out.x = points.x
    out.y = points.y
    out.z = points.z

    src_names = points.array.dtype.names
    dst_names = out.array.dtype.names

    for name in dst_names:
        if name in ("X", "Y", "Z"):
            continue
        if name in src_names:
            out[name] = points[name]

    return out

In [3]:
def merge_las_streaming(input_files, output_path, chunk_size=2_000_000, force=False):
    input_files = [Path(p) for p in input_files]
    output_path = Path(output_path)

    if output_path.exists() and not force:
        print(f"[SKIP] merge déjà existant : {output_path}")
        return output_path

    check_inputs_compatible(input_files)

    with laspy.open(input_files[0]) as r0:
        out_header = copy.deepcopy(r0.header)

    tmp_out = output_path.with_suffix(".tmp.las")
    if tmp_out.exists():
        tmp_out.unlink()

    total_written = 0
    t0 = time.time()

    with laspy.open(tmp_out, mode="w", header=out_header) as writer:
        for i, in_path in enumerate(input_files, start=1):
            print(f"[MERGE] {i}/{len(input_files)} : {in_path}")
            with laspy.open(in_path) as reader:
                n_file = 0
                for pts in reader.chunk_iterator(chunk_size):
                    out_pts = remap_points_to_target_header(pts, out_header)
                    writer.write_points(out_pts)
                    n = len(out_pts)
                    n_file += n
                    total_written += n
                print(f"  -> points écrits : {n_file:,}")

    os.replace(tmp_out, output_path)

    print(f"\n[OK] merge terminé : {output_path}")
    print(f"     total points : {total_written:,}")
    print(f"     temps        : {(time.time()-t0)/60:.1f} min")

    return output_path


MERGED_ALLER = merge_las_streaming(
    input_files=ALLER_FILES,
    output_path=MERGED_ALLER,
    chunk_size=CHUNK_SIZE,
    force=FORCE_REBUILD,
)

[MERGE] 1/2 : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/merged/ALL/merged_1000_VUX_PUCK.las
  -> points écrits : 294,117,790
[MERGE] 2/2 : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/merged/ALL/merged_2000_VUX_PUCK.las
  -> points écrits : 66,950,257

[OK] merge terminé : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/merged/aller_merged.las
     total points : 361,068,047
     temps        : 6.0 min


In [4]:
def crop_las_by_gps_time(input_path, output_path, tmin, tmax, chunk_size=2_000_000, force=False):
    input_path = Path(input_path)
    output_path = Path(output_path)

    if output_path.exists() and not force:
        print(f"[SKIP] crop déjà existant : {output_path}")
        return output_path

    if tmax < tmin:
        tmin, tmax = tmax, tmin

    with laspy.open(input_path) as reader:
        header = copy.deepcopy(reader.header)
        if "gps_time" not in dim_names(header):
            raise ValueError(f"'gps_time' absent de {input_path}")

    tmp_out = output_path.with_suffix(".tmp.las")
    if tmp_out.exists():
        tmp_out.unlink()

    total_in = 0
    total_out = 0
    t0 = time.time()

    print(f"\n[CROP] {input_path}")
    print(f"       fenêtre : [{tmin:.3f}, {tmax:.3f}]")

    with laspy.open(input_path) as reader, laspy.open(tmp_out, mode="w", header=header) as writer:
        for pts in reader.chunk_iterator(chunk_size):
            gt = pts.gps_time
            total_in += len(gt)
            mask = (gt >= tmin) & (gt <= tmax)

            if np.any(mask):
                sub = pts[mask]
                writer.write_points(sub)
                total_out += len(sub)

    os.replace(tmp_out, output_path)

    print(f"[OK] crop écrit : {output_path}")
    print(f"     points in  : {total_in:,}")
    print(f"     points out : {total_out:,}")
    print(f"     temps      : {(time.time()-t0)/60:.1f} min")

    return output_path


CROPPED_ALLER = crop_las_by_gps_time(
    input_path=MERGED_ALLER,
    output_path=CROPPED_ALLER,
    tmin=ALLER_TMIN,
    tmax=ALLER_TMAX,
    chunk_size=CHUNK_SIZE,
    force=FORCE_REBUILD,
)


[CROP] /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/merged/aller_merged.las
       fenêtre : [305110.000, 305710.000]
[OK] crop écrit : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/cropped/aller_crop.las
     points in  : 361,068,047
     points out : 305,874,283
     temps      : 4.9 min


In [5]:
for p in [MERGED_ALLER, CROPPED_ALLER, RETOUR_FILE]:
    print_las_info(p)
    tmin, tmax, n = stream_las_time_bounds(p, chunk_size=CHUNK_SIZE)
    print(f"  gps_time min/max : {tmin:.3f} -> {tmax:.3f}")
    print(f"  count (stream)   : {n:,}")


/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/merged/aller_merged.las
  point_count : 361,068,047
  point_format: 1
  scales      : [0.001 0.001 0.001]
  offsets     : [2.542171e+06 1.156820e+06 8.820000e+02]
  file size   : 13.79 GB
  dims        : ['X', 'Y', 'Z', 'intensity', 'return_number', 'number_of_returns', 'scan_direction_flag', 'edge_of_flight_line', 'classification', 'synthetic', 'key_point', 'withheld', 'scan_angle_rank', 'user_data', 'point_source_id', 'gps_time', 'lasvec_x', 'lasvec_y', 'lasvec_z', 'scanner_src']
  gps_time min/max : 305060.000 -> 305375.884
  count (stream)   : 361,068,047

/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/cropped/aller_crop.las
  point_count : 305,874,283
  point_format: 1
  scales      : [0.001 0.001 0.001]
  offsets     : [2.542171e+06 1.156820e+06 8.820000e+02]
  file size   : 11.68 GB
  dims        : ['X', 'Y', 'Z', 'intensity', 'return_number', 'number_of_returns', 'scan_direction_fl

In [6]:
print("Fichiers à donner à patcher :")
print("  aller crop :", CROPPED_ALLER)
print("  retour full:", RETOUR_FILE)

Fichiers à donner à patcher :
  aller crop : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/cropped/aller_crop.las
  retour full: /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/merged/ALL/merged_3000_VUX_PUCK.las


In [8]:
# ============================================================
# INPUTS FOR VERTICAL EASTING SPLIT
# ============================================================

CROPPED_FILES = sorted(CROP_DIR.glob("*.las"))

print("LAS trouvés dans cropped/:")
for p in CROPPED_FILES:
    print("  ", p)

if len(CROPPED_FILES) != 2:
    raise ValueError(
        f"J'attendais exactement 2 LAS dans {CROP_DIR}, trouvé {len(CROPPED_FILES)}.\n"
        "Si besoin, renseigne-les manuellement dans SPLIT_INPUT_A et SPLIT_INPUT_B."
    )

SPLIT_INPUT_A = CROPPED_FILES[0]
SPLIT_INPUT_B = CROPPED_FILES[1]

VERT_SPLIT_DIR = WORK_ROOT / "split_vertical_easting"
VERT_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("\nSPLIT_INPUT_A:", SPLIT_INPUT_A)
print("SPLIT_INPUT_B:", SPLIT_INPUT_B)
print("VERT_SPLIT_DIR:", VERT_SPLIT_DIR)

LAS trouvés dans cropped/:
   /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/cropped/aller_crop.las


ValueError: J'attendais exactement 2 LAS dans /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/cropped, trouvé 1.
Si besoin, renseigne-les manuellement dans SPLIT_INPUT_A et SPLIT_INPUT_B.

In [9]:
def stream_x_bounds(path: Path, chunk_size=2_000_000):
    xmin = np.inf
    xmax = -np.inf
    total = 0

    with laspy.open(path) as r:
        for pts in r.chunk_iterator(chunk_size):
            if len(pts) == 0:
                continue
            x = pts.x
            xmin = min(xmin, float(np.min(x)))
            xmax = max(xmax, float(np.max(x)))
            total += len(x)

    return xmin, xmax, total


def split_las_by_x_edges(input_path, out_paths, x_edges, chunk_size=2_000_000):
    input_path = Path(input_path)
    out_paths = [Path(p) for p in out_paths]

    with laspy.open(input_path) as r:
        headers = [copy.deepcopy(r.header) for _ in out_paths]

    writers = []
    try:
        for out_path, hdr in zip(out_paths, headers):
            out_path.parent.mkdir(parents=True, exist_ok=True)
            writers.append(laspy.open(out_path, mode="w", header=hdr))

        counts = [0] * len(out_paths)

        with laspy.open(input_path) as r:
            for pts in r.chunk_iterator(chunk_size):
                x = pts.x
                bin_ids = np.digitize(x, x_edges, right=False)

                for i, w in enumerate(writers):
                    mask = (bin_ids == i)
                    if np.any(mask):
                        sub = pts[mask]
                        w.write_points(sub)
                        counts[i] += len(sub)

        return counts

    finally:
        for w in writers:
            w.close()


def split_pair_vertical_easting(path_a, path_b, out_root, n_parts=3, chunk_size=2_000_000):
    path_a = Path(path_a)
    path_b = Path(path_b)
    out_root = Path(out_root)

    if n_parts != 3:
        raise ValueError("Ici la version est prévue pour 3 parties.")

    xmin_a, xmax_a, na = stream_x_bounds(path_a, chunk_size=chunk_size)
    xmin_b, xmax_b, nb = stream_x_bounds(path_b, chunk_size=chunk_size)

    xmin = min(xmin_a, xmin_b)
    xmax = max(xmax_a, xmax_b)

    if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
        raise ValueError("Bornes X invalides.")

    width = (xmax - xmin) / 3.0
    x1 = xmin + width
    x2 = xmin + 2.0 * width
    x_edges = [x1, x2]

    print("Bornes globales Easting:")
    print(f"  xmin = {xmin:.3f}")
    print(f"  xmax = {xmax:.3f}")
    print("Seuils utilisés pour les 2 nuages:")
    print(f"  x1   = {x1:.3f}")
    print(f"  x2   = {x2:.3f}")

    pair_dir = out_root / f"{path_a.stem}__{path_b.stem}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    out_a = [pair_dir / f"part_{i+1}" / path_a.name for i in range(n_parts)]
    out_b = [pair_dir / f"part_{i+1}" / path_b.name for i in range(n_parts)]

    print("\n[SPLIT] écriture A...")
    counts_a = split_las_by_x_edges(path_a, out_a, x_edges=x_edges, chunk_size=chunk_size)

    print("[SPLIT] écriture B...")
    counts_b = split_las_by_x_edges(path_b, out_b, x_edges=x_edges, chunk_size=chunk_size)

    df = pd.DataFrame({
        "part": [f"part_{i+1}" for i in range(n_parts)],
        "x_min": [xmin, x1, x2],
        "x_max": [x1, x2, xmax],
        "file_a": [str(p) for p in out_a],
        "file_b": [str(p) for p in out_b],
        "count_a": counts_a,
        "count_b": counts_b,
    })

    df.to_csv(pair_dir / "split_manifest.csv", index=False)

    print(f"\n[OK] split vertical terminé dans : {pair_dir}")
    return pair_dir, df

In [ ]:
pair_dir, split_df = split_pair_vertical_easting(
    path_a=SPLIT_INPUT_A,
    path_b=SPLIT_INPUT_B,
    out_root=VERT_SPLIT_DIR,
    n_parts=3,
    chunk_size=CHUNK_SIZE,
)

split_df

Bornes globales Easting:
  xmin = 2542558.405
  xmax = 2546339.724
Seuils utilisés pour les 2 nuages:
  x1   = 2543818.845
  x2   = 2545079.284

[SPLIT] écriture A...
[SPLIT] écriture B...

[OK] split vertical terminé dans : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_2


,part,x_min,x_max,file_a,file_b,count_a,count_b
0,part_1,2.542558e+06,2.543819e+06,/media/b085164/Elements/CALIB_26_02_25/georef_...,/media/b085164/Elements/CALIB_26_02_25/georef_...,113225598,107154050
1,part_2,2.543819e+06,2.545079e+06,/media/b085164/Elements/CALIB_26_02_25/georef_...,/media/b085164/Elements/CALIB_26_02_25/georef_...,100507267,110049372
2,part_3,2.545079e+06,2.546340e+06,/media/b085164/Elements/CALIB_26_02_25/georef_...,/media/b085164/Elements/CALIB_26_02_25/georef_...,92141369,109437691


In [ ]:
print("Paires générées :\n")

for part_dir in sorted(pair_dir.glob("part_*")):
    las_files = sorted(part_dir.glob("*.las"))
    print(part_dir)
    for p in las_files:
        print("   ", p)
    print()

Paires générées :

/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_2/part_1
    /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_2/part_1/patch_1.las
    /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_2/part_1/patch_2.las

/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_2/part_2
    /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_2/part_2/patch_1.las
    /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_2/part_2/patch_2.las

/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_2/part_3
    /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting/patch_1__patch_

## Enlever puck des 3 scanners

In [1]:
from pathlib import Path
import copy
import os
import time
import numpy as np
import laspy

# ============================================================
# CONFIG
# ============================================================

SRC_ROOT = Path("/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting")
DST_ROOT = Path("/media/b085164/Elements/CALIB_26_02_25/georef_VUX_traj_ODyN/Patcher_long_outage/split_vertical_easting")

CHUNK_SIZE = 2_000_000
SCANNER_SRC_KEEP = 2
FORCE_OVERWRITE = False

DST_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def dim_names(header):
    return list(header.point_format.dimension_names)

def filter_las_scanner_src(
    in_path: Path,
    out_path: Path,
    scanner_src_keep: int = 2,
    chunk_size: int = 2_000_000,
    force_overwrite: bool = False,
):
    if out_path.exists() and not force_overwrite:
        print(f"[SKIP] already exists: {out_path}")
        return

    out_path.parent.mkdir(parents=True, exist_ok=True)

    with laspy.open(in_path) as reader:
        header = copy.deepcopy(reader.header)
        dims = dim_names(header)

        if "scanner_src" not in dims:
            raise ValueError(f"'scanner_src' absent dans {in_path}")

    tmp_out = out_path.with_suffix(".tmp.las")
    if tmp_out.exists():
        tmp_out.unlink()

    total_in = 0
    total_out = 0
    t0 = time.time()

    with laspy.open(in_path) as reader, laspy.open(tmp_out, mode="w", header=header) as writer:
        for pts in reader.chunk_iterator(chunk_size):
            total_in += len(pts)

            mask = (pts.scanner_src == scanner_src_keep)
            if np.any(mask):
                sub = pts[mask]
                writer.write_points(sub)
                total_out += len(sub)

    os.replace(tmp_out, out_path)

    dt = time.time() - t0
    print(
        f"[OK] {in_path.relative_to(SRC_ROOT)} | "
        f"in={total_in:,} | out={total_out:,} | "
        f"time={dt:.1f}s"
    )

# ============================================================
# RUN
# ============================================================

las_files = sorted(SRC_ROOT.rglob("*.las"))

if not las_files:
    raise FileNotFoundError(f"Aucun fichier .las trouvé sous {SRC_ROOT}")

print(f"Source root : {SRC_ROOT}")
print(f"Dest root   : {DST_ROOT}")
print(f"LAS found   : {len(las_files)}")
print(f"Keep scanner_src = {SCANNER_SRC_KEEP}")
print("-" * 60)

errors = []

for in_path in las_files:
    try:
        rel = in_path.relative_to(SRC_ROOT)
        out_path = DST_ROOT / rel
        filter_las_scanner_src(
            in_path=in_path,
            out_path=out_path,
            scanner_src_keep=SCANNER_SRC_KEEP,
            chunk_size=CHUNK_SIZE,
            force_overwrite=FORCE_OVERWRITE,
        )
    except Exception as e:
        print(f"[FAIL] {in_path}: {type(e).__name__}: {e}")
        errors.append((in_path, str(e)))

print("-" * 60)
print(f"Done. Success: {len(las_files) - len(errors)} | Failures: {len(errors)}")

if errors:
    print("\nFailures:")
    for p, err in errors:
        print(f" - {p}: {err}")

Source root : /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_ODyN/Patcher/split_vertical_easting
Dest root   : /media/b085164/Elements/CALIB_26_02_25/georef_VUX_traj_ODyN/Patcher_long_outage/split_vertical_easting
LAS found   : 6
Keep scanner_src = 2
------------------------------------------------------------
[OK] Flights_1_2/Patch_from_scan_1_with_2.las | in=113,225,598 | out=101,488,164 | time=111.7s
[OK] Flights_1_2/Patch_from_scan_2_with_1.las | in=107,154,050 | out=96,596,985 | time=119.9s
[OK] Flights_3_4/Patch_from_scan_3_with_4.las | in=100,507,267 | out=90,803,341 | time=111.4s
[OK] Flights_3_4/Patch_from_scan_4_with_3.las | in=110,049,372 | out=99,213,345 | time=83.8s
[OK] Flights_5_6/Patch_from_scan_5_with_6.las | in=92,141,369 | out=82,336,468 | time=63.9s
[OK] Flights_5_6/Patch_from_scan_6_with_5.las | in=109,437,691 | out=96,259,527 | time=129.0s
------------------------------------------------------------
Done. Success: 6 | Failures: 0


## Preparer data Patcher

In [11]:
from pathlib import Path
import shutil
import numpy as np
import laspy

# ============================================================
# PATHS
# ============================================================
root = Path("/media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1")
cropped_dir = root / "Patcher" / "cropped"
pairs_root = cropped_dir / "patcher_pairs"

aller_crop_path = cropped_dir / "aller_crop.las"
retour_in_path = root / "merged" / "ALL" / "merged_3000_VUX_PUCK.las"
retour_crop_path = cropped_dir / "retour_crop.las"

pairs_root.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
time_max = 305710.0
split_axis = "x"      # "x" = bandes verticales, "y" = bandes horizontales
n_strips = 3
chunk_size = 2_000_000
sample_every = 50     # pour estimer les quantiles sans tout charger
min_points_warn = 50_000

# convention de nommage Patcher
pair_ids = [(1, 2), (3, 4), (5, 6)]

# ============================================================
# HELPERS
# ============================================================
def crop_las_by_time(in_path: Path, out_path: Path, time_max: float, chunk_size=1_000_000):
    n_written = 0

    if out_path.exists():
        out_path.unlink()

    with laspy.open(in_path) as src:
        header = clone_las_header(src.header)

        with laspy.open(out_path, mode="w", header=header) as dst:
            for points in src.chunk_iterator(chunk_size):
                gps_time = points["gps_time"]
                mask = gps_time <= time_max

                if np.any(mask):
                    sub = points[mask]
                    dst.write_points(sub)
                    n_written += len(sub)

    return n_written
import copy
import laspy

def clone_las_header(header):
    new_header = laspy.LasHeader(
        point_format=header.point_format,
        version=header.version
    )
    new_header.scales = header.scales
    new_header.offsets = header.offsets

    # préserver VLRs / CRS / extra bytes si présents
    try:
        new_header.vlrs = copy.deepcopy(header.vlrs)
    except Exception:
        pass

    try:
        new_header.evlrs = copy.deepcopy(header.evlrs)
    except Exception:
        pass

    return new_header

def get_bbox(las_path: Path):
    with laspy.open(las_path) as f:
        mins = f.header.mins
        maxs = f.header.maxs

    return {
        "xmin": float(mins[0]),
        "ymin": float(mins[1]),
        "xmax": float(maxs[0]),
        "ymax": float(maxs[1]),
    }


def intersect_bbox(b1, b2):
    xmin = max(b1["xmin"], b2["xmin"])
    ymin = max(b1["ymin"], b2["ymin"])
    xmax = min(b1["xmax"], b2["xmax"])
    ymax = min(b1["ymax"], b2["ymax"])

    if xmin >= xmax or ymin >= ymax:
        raise ValueError("Pas d'intersection spatiale entre aller_crop et retour_crop.")

    return {"xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax}


def sample_axis_in_bbox(
    las_path: Path,
    bbox: dict,
    axis="x",
    chunk_size=1_000_000,
    sample_every=50,
):
    vals = []

    with laspy.open(las_path) as src:
        for points in src.chunk_iterator(chunk_size):
            x = points.x
            y = points.y

            mask = (
                (x >= bbox["xmin"]) & (x <= bbox["xmax"]) &
                (y >= bbox["ymin"]) & (y <= bbox["ymax"])
            )

            if not np.any(mask):
                continue

            arr = x[mask] if axis == "x" else y[mask]
            vals.append(arr[::sample_every])

    if not vals:
        return np.array([])

    return np.concatenate(vals)


def build_strip_bbox(common_bbox, axis, a0, a1):
    if axis == "x":
        return {
            "xmin": a0,
            "xmax": a1,
            "ymin": common_bbox["ymin"],
            "ymax": common_bbox["ymax"],
        }
    elif axis == "y":
        return {
            "xmin": common_bbox["xmin"],
            "xmax": common_bbox["xmax"],
            "ymin": a0,
            "ymax": a1,
        }
    else:
        raise ValueError("axis must be 'x' or 'y'")


def write_spatial_subset_las(in_path: Path, out_path: Path, bbox: dict, chunk_size=1_000_000):
    n_written = 0

    if out_path.exists():
        out_path.unlink()

    with laspy.open(in_path) as src:
        header = clone_las_header(src.header)

        with laspy.open(out_path, mode="w", header=header) as dst:
            for points in src.chunk_iterator(chunk_size):
                x = points.x
                y = points.y

                mask = (
                    (x >= bbox["xmin"]) & (x <= bbox["xmax"]) &
                    (y >= bbox["ymin"]) & (y <= bbox["ymax"])
                )

                if np.any(mask):
                    sub = points[mask]
                    dst.write_points(sub)
                    n_written += len(sub)

    return n_written


# ============================================================
# 1) CROP RETOUR BY TIME
# ============================================================
print("Cropping retour by gps_time...")
n_retour_crop = crop_las_by_time(
    in_path=retour_in_path,
    out_path=retour_crop_path,
    time_max=time_max,
    chunk_size=chunk_size,
)
print(f"  retour_crop written: {retour_crop_path}")
print(f"  points kept: {n_retour_crop:,}")

# ============================================================
# 2) COMMON SPATIAL OVERLAP
# ============================================================
bbox_aller = get_bbox(aller_crop_path)
bbox_retour = get_bbox(retour_crop_path)
common_bbox = intersect_bbox(bbox_aller, bbox_retour)

print("\nCommon overlap bbox:")
for k, v in common_bbox.items():
    print(f"  {k}: {v:.3f}")

# ============================================================
# 3) COMPUTE 3 VERTICAL STRIPS WITH QUANTILES
#    (roughly similar sizes)
# ============================================================
print("\nSampling overlap to estimate strip boundaries...")
s1 = sample_axis_in_bbox(
    aller_crop_path,
    common_bbox,
    axis=split_axis,
    chunk_size=chunk_size,
    sample_every=sample_every,
)
s2 = sample_axis_in_bbox(
    retour_crop_path,
    common_bbox,
    axis=split_axis,
    chunk_size=chunk_size,
    sample_every=sample_every,
)

samples = np.concatenate([s1, s2]) if len(s1) and len(s2) else (s1 if len(s1) else s2)

if len(samples) == 0:
    raise ValueError("Aucun point trouvé dans l'overlap pour définir les strips.")

q = np.linspace(0, 1, n_strips + 1)
edges = np.quantile(samples, q)

# force exact outer bounds
if split_axis == "x":
    edges[0] = common_bbox["xmin"]
    edges[-1] = common_bbox["xmax"]
else:
    edges[0] = common_bbox["ymin"]
    edges[-1] = common_bbox["ymax"]

# sécurité si deux quantiles sont trop proches
for i in range(1, len(edges)):
    if edges[i] <= edges[i - 1]:
        edges[i] = edges[i - 1] + 1e-6

print("\nStrip edges:")
for i, e in enumerate(edges):
    print(f"  edge_{i}: {e:.3f}")

# ============================================================
# 4) WRITE PATCHER-LIKE PAIRS
# ============================================================
print("\nWriting Patcher-like pairs...")

summary = []

for i in range(n_strips):
    scan_a, scan_b = pair_ids[i]

    a0 = edges[i]
    a1 = edges[i + 1]
    strip_bbox = build_strip_bbox(common_bbox, split_axis, a0, a1)

    flight_dir = pairs_root / f"Flights_{scan_a}_{scan_b}"
    flight_dir.mkdir(parents=True, exist_ok=True)

    aller_out = flight_dir / f"Patch_from_scan_{scan_a}_with_{scan_b}.las"
    retour_out = flight_dir / f"Patch_from_scan_{scan_b}_with_{scan_a}.las"

    n1 = write_spatial_subset_las(
        in_path=aller_crop_path,
        out_path=aller_out,
        bbox=strip_bbox,
        chunk_size=chunk_size,
    )
    n2 = write_spatial_subset_las(
        in_path=retour_crop_path,
        out_path=retour_out,
        bbox=strip_bbox,
        chunk_size=chunk_size,
    )

    summary.append((flight_dir.name, n1, n2))

    print(f"\n{flight_dir.name}")
    print(f"  {aller_out.name}:  {n1:,} pts")
    print(f"  {retour_out.name}: {n2:,} pts")

    if n1 < min_points_warn or n2 < min_points_warn:
        print("  WARNING: one file has few points")

print("\nDone.")
print(f"\nretour cropped file: {retour_crop_path}")
print(f"patcher-like pairs root: {pairs_root}")

print("\nSummary:")
for name, n1, n2 in summary:
    print(f"  {name}: {n1:,} / {n2:,} pts")

Cropping retour by gps_time...
  retour_crop written: /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/cropped/retour_crop.las
  points kept: 326,641,135

Common overlap bbox:
  xmin: 2542653.190
  ymin: 1157301.810
  xmax: 2546326.806
  ymax: 1158646.054

Sampling overlap to estimate strip boundaries...

Strip edges:
  edge_0: 2542653.190
  edge_1: 2543779.962
  edge_2: 2545038.053
  edge_3: 2546326.806

Writing Patcher-like pairs...

Flights_1_2
  Patch_from_scan_1_with_2.las:  103,317,167 pts
  Patch_from_scan_2_with_1.las: 104,935,550 pts

Flights_3_4
  Patch_from_scan_3_with_4.las:  99,697,795 pts
  Patch_from_scan_4_with_3.las: 108,558,308 pts

Flights_5_6
  Patch_from_scan_5_with_6.las:  95,275,488 pts
  Patch_from_scan_6_with_5.las: 112,975,518 pts

Done.

retour cropped file: /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outage_1/Patcher/cropped/retour_crop.las
patcher-like pairs root: /media/b085164/Elements/CALIB_26_02_25/georef_ALL_traj_outag